In [2]:
import zipfile
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, balanced_accuracy_score
from torch.utils.data import DataLoader, TensorDataset

c:\Users\Kacper\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [32]:
label_sale_price = lambda amount: 0 if amount <= 100000 else 2 if amount > 350000 else 1

train_data, test_data = (
    pd.read_csv('train_data.csv'),
    pd.read_csv('test_data.csv'),
)

In [33]:
# create class and delete SalePrice

train_data['Target'] = train_data['SalePrice'].apply(label_sale_price)

train_data = train_data.drop('SalePrice', axis=1)

train_data

,YearBuilt,Size(sqf),Floor,HallwayType,HeatingType,AptManageType,N_Parkinglot(Ground),N_Parkinglot(Basement),TimeToBusStop,TimeToSubway,N_manager,N_elevators,SubwayStation,N_FacilitiesInApt,N_FacilitiesNearBy(Total),N_SchoolNearBy(Total),Target
0,2006,814,3,terraced,individual_heating,management_in_trust,111.0,184.0,5min~10min,10min~15min,3.0,0.0,Kyungbuk_uni_hospital,5,6.0,9.0,1
1,1985,587,8,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0,0
2,1985,587,6,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0,0
3,2006,2056,8,terraced,individual_heating,management_in_trust,249.0,536.0,0~5min,0-5min,5.0,11.0,Sin-nam,5,3.0,7.0,2
4,1992,644,2,mixed,individual_heating,self_management,142.0,79.0,5min~10min,15min~20min,4.0,8.0,Myung-duk,3,9.0,14.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4119,2007,1928,24,terraced,individual_heating,management_in_trust,0.0,1270.0,0~5min,0-5min,14.0,16.0,Kyungbuk_uni_hospital,10,9.0,10.0,2
4120,2015,644,22,terraced,individual_heating,management_in_trust,102.0,400.0,0~5min,5min~10min,5.0,10.0,Daegu,7,7.0,11.0,1
4121,2007,868,20,terraced,individual_heating,management_in_trust,0.0,1270.0,0~5min,0-5min,14.0,16.0,Kyungbuk_uni_hospital,10,9.0,10.0,2
4122,1978,1327,1,corridor,individual_heating,self_management,87.0,0.0,0~5min,0-5min,1.0,4.0,Kyungbuk_uni_hospital,3,7.0,11.0,1


In [34]:
# target variable
y_train = train_data['Target']
X_train = train_data.drop('Target', axis=1)
X_test = test_data.copy()

# concat sets to link categories in the same manner
train_len = len(train_data)
all_data = pd.concat([X_train, X_test], axis=0).reset_index(drop=True)

all_data

,YearBuilt,Size(sqf),Floor,HallwayType,HeatingType,AptManageType,N_Parkinglot(Ground),N_Parkinglot(Basement),TimeToBusStop,TimeToSubway,N_manager,N_elevators,SubwayStation,N_FacilitiesInApt,N_FacilitiesNearBy(Total),N_SchoolNearBy(Total)
0,2006,814,3,terraced,individual_heating,management_in_trust,111.0,184.0,5min~10min,10min~15min,3.0,0.0,Kyungbuk_uni_hospital,5,6.0,9.0
1,1985,587,8,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0
2,1985,587,6,corridor,individual_heating,self_management,80.0,76.0,0~5min,5min~10min,2.0,2.0,Daegu,3,12.0,4.0
3,2006,2056,8,terraced,individual_heating,management_in_trust,249.0,536.0,0~5min,0-5min,5.0,11.0,Sin-nam,5,3.0,7.0
4,1992,644,2,mixed,individual_heating,self_management,142.0,79.0,5min~10min,15min~20min,4.0,8.0,Myung-duk,3,9.0,14.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5886,2006,2056,2,terraced,individual_heating,management_in_trust,249.0,536.0,0~5min,0-5min,5.0,11.0,Sin-nam,5,3.0,7.0
5887,2007,1394,7,terraced,individual_heating,management_in_trust,554.0,524.0,0~5min,0-5min,5.0,10.0,Banwoldang,4,9.0,8.0
5888,1993,644,20,mixed,individual_heating,management_in_trust,523.0,536.0,0~5min,15min~20min,8.0,20.0,Myung-duk,4,14.0,17.0
5889,2008,914,11,terraced,individual_heating,management_in_trust,197.0,475.0,5min~10min,0-5min,6.0,14.0,Sin-nam,8,7.0,9.0


In [35]:
# filling numeric columns with median and text with None

numeric_cols    = all_data.select_dtypes(include=['int64', 'float64']).columns
categorial_cols = all_data.select_dtypes(include=['object']).columns

all_data[numeric_cols] = all_data[numeric_cols].fillna(
    all_data[numeric_cols].median()
)

all_data[categorial_cols] = all_data[categorial_cols].fillna('None')


C:\Users\Kacper\AppData\Local\Temp\ipykernel_19700\1297411525.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorial_cols = all_data.select_dtypes(include=['object']).columns


In [36]:
# hot one encoding, cleaning up and data scaling for MLP

all_data = pd.get_dummies(all_data)

scaler = StandardScaler()

X_train_clean = all_data.iloc[:train_len, :]
X_test_clean = all_data.iloc[train_len:, :]

X_train_scaled = scaler.fit_transform(X_train_clean)
X_test_scaled = scaler.transform(X_test_clean)

In [37]:
class HousePriceNet(nn.Module):
    def __init__(self, input_size):
        super(HousePriceNet, self).__init__()

        self.net = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.01),
            nn.Dropout(0.2),

            nn.Linear(64, 3)    # 3 because (0, 1, 2) = (cheap, medium, expensive)
        )

    def forward(self, x):
        return self.net(x)

In [38]:
X_tensor, y_tensor = (
    torch.FloatTensor(X_train_scaled),
    torch.LongTensor(y_train.values)
)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_tensor = torch.FloatTensor(class_weights)

weights_tensor

tensor([2.4460, 0.4594, 2.4117])

In [39]:
num_cols = X_train_scaled.shape[1]
model = HousePriceNet(input_size=num_cols)

# loss function
criterion = nn.CrossEntropyLoss(weight=weights_tensor)

# adam optimizer
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# learn loop
epochs = 50
model.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)
        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss = {epoch_loss/len(dataloader):.4f}")

print("Training complete")

Epoch [10/50], Loss = 0.3299
Epoch [20/50], Loss = 0.3042
Epoch [30/50], Loss = 0.3085
Epoch [40/50], Loss = 0.3028
Epoch [50/50], Loss = 0.2998
Training complete


In [ ]:
model.eval()

X_test_tensor = torch.FloatTensor(X_test_scaled)

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predictions = torch.max(test_outputs, 1)

final_preds = predictions.numpy()

# save predictions to pred.csv (no header, one integer column)
pd.DataFrame(final_preds).to_csv('pred.csv', index=False, header=False)

print(f"Saved {len(final_preds)} predictions to pred.csv")
print(f"Class distribution: {pd.Series(final_preds).value_counts().sort_index().to_dict()}")


Saved 1767 predictions to pred.csv
Class distribution: {0: 405, 1: 977, 2: 385}


In [ ]:
from sklearn.metrics import classification_report

model.eval()

with torch.no_grad():
    train_outputs = model(X_tensor)
    _, train_predictions = torch.max(train_outputs, 1)

train_preds_np = train_predictions.numpy()

print("\n--- REPORT AFTER TRAINING ---")
print("Balanced accuracy across all classes:")
print(f"{balanced_accuracy_score(y_train, train_preds_np) * 100:.2f}%\n")

print("Detailed classification report (pay attention to the 'recall' column):")
print(classification_report(y_train, train_preds_np))



--- RAPORT PO TRENINGU ---
Średnia dokładność (Balanced Accuracy) dla każdej z klas:
89.54%

Szczegółowy raport klasyfikacji (zwróć uwagę na kolumnę 'recall'):
              precision    recall  f1-score   support

           0       0.59      0.99      0.74       562
           1       0.99      0.74      0.84      2992
           2       0.57      0.95      0.72       570

    accuracy                           0.80      4124
   macro avg       0.72      0.90      0.77      4124
weighted avg       0.88      0.80      0.81      4124



In [3]:
# pack into zip file

DAY = "sroda"
AUTHOR_1 = "BagińskiJakub"
AUTHOR_2 = "GórskiKacper"
CODE_FILE = "cw2.ipynb"

autorzy = sorted([AUTHOR_1, AUTHOR_2])
nazwa_zip = f"{DAY}_{autorzy[0]}_{autorzy[1]}.zip"

with zipfile.ZipFile(nazwa_zip, 'w') as zipf:
    if os.path.exists('pred.csv'):
        zipf.write('pred.csv')
    else:
        print("ERROR: pred.csv not found!")

    if os.path.exists(CODE_FILE):
        zipf.write(CODE_FILE)
    else:
        print(f"ERROR: {CODE_FILE} not found!")

print(f"Package ready: {nazwa_zip}")


Package ready: sroda_BagińskiJakub_GórskiKacper.zip
